In [1]:
import gzip
import pickle
import numpy as np
import pandas as pd

In [ ]:
# Select a model
models_list = ['DNABERT', 'nucleotide-transformer']
model = 'DNABERT'

# Select a submodel
submodels_list = ['finetuned', 'pretrained', 'random_init', 'random_pretrained'] #NOTE - random_pretrained is only available for DNABERT
submodel = 'finetuned'

# Select a dataset
dataset = 'fake_TATA'

# Number of layers
nlayers = 12 if model == 'DNABERT' else 29
nheads = 12 if model == 'DNABERT' else 16

# Pathways are highlighted with the hastag #REVIEW

In [ ]:

def kmer2seq(kmers_list:list) -> str:
    first:bool = True
    for token in kmers_list:
        if first:
            sequence:str = token
            first = False
        elif (token != '[SEP]' and token != '[PAD]'):
            sequence += token[-1]
        else:
            break
    return(sequence)

sequences_coordinates:pd.DataFrame = pd.read_csv(f'test_datasets/fake_TATA_test.csv')
# Create a dictionary with all the data
results_dict:dict = {}
for layer in range(nlayers):
    with open(f'{model}/attention_scores/{submodel}/layer{layer}.p', 'rb') as f: #REVIEW - Check that the path is correct
        results:dict = pickle.load(f)
    for head in range(nheads):
        for example in range(len(results[head])):
            if model == "DNABERT":
                sequence:str = kmer2seq(results[head][example][1])
            else:
                sequence:str = ''.join([kmer for kmer in results[head][example][1] if (kmer != '<cls>' and kmer != '<pad>')])
            if layer == 0 and head ==0:
                results_dict[sequence] = {}
                results_dict[sequence]['kmers'] = [kmer for kmer in results[head][example][1] if (kmer not in ['[SEP]', '[PAD]', '<cls>', '<pad>'])]
            kmers_vector_length = len(results_dict[sequence]['kmers'])
            results_dict[sequence][f'layer{layer}-head{head}'] = np.array(results[head][example][0][1:kmers_vector_length+1]) #The first score belong to a special token

# Annotate GC content
for sequence in results_dict:
    kmer_list:list = results_dict[sequence]['kmers']
    gc_vector:list = []
    for kmer in kmer_list:
        ## Change C and G to 1, and A and T to 0
        gc_sequence = kmer.replace('C', '1').replace('G', '1').replace('A', '0').replace('T', '0')
        gc_sequence = [int(i) for i in gc_sequence]
        gc_vector.append(np.mean(gc_sequence))
    
    ## Add this layer to the original one
    results_dict[sequence]['GC'] = np.array(gc_vector)
del(sequence, kmer_list, gc_vector, kmer, gc_sequence)

# Add TATAAA and ATATAA kmer label
for sequence in results_dict:
    results_dict[sequence]['TATAAA'] = np.array([1 if kmer == 'TATAAA' else 0 for kmer in results_dict[sequence]['kmers']])
    results_dict[sequence]['ATATAA'] = np.array([1 if kmer == 'ATATAA' else 0 for kmer in results_dict[sequence]['kmers']])

del(sequence)

# Add sequence label
for sequence in results_dict.keys():
    kmers_vector_length:int = len(results_dict[sequence]['kmers'])
    results_dict[sequence]['label'] = np.full(kmers_vector_length, sequences_coordinates.loc[sequences_coordinates["Sequence"] == sequence, "Label"].iloc[0])

del(sequence)

# Convert dictionary into list of dictionaries
df:list = []
for key, subdict in results_dict.items():
    dict_by_row:dict = {'sequence': key}
    for subkey, values_list in subdict.items():
        dict_by_row[subkey] = ','.join(map(str, values_list))
    df.append(dict_by_row)

# Create DataFrame
df:pd.DataFrame = pd.DataFrame(df)

# Save the dataframe as a csv file with semicolon separator
with gzip.open(f'{model}/{model}_{submodel}_{dataset}_scores_corformat.csv.gz', 'wt', newline='', encoding='utf-8') as f: #REVIEW - Check that the path is correct
    df.to_csv(f, sep=';', index=False)

del(key, subdict, subkey, dict_by_row, values_list, df, f)